# RoBERTa Multi-Class Diagnosis Models

Trains **two** RoBERTa classifiers on `suicidewatch_filtered_df`:

1. **3-class severity model** — predicts `threeclass_label`.
2. **Subreddit model** — predicts `multiclass_label` (after dropping ultra-rare classes).

Each model gets its own 80/10/10 stratified split, class-weighted loss, and checkpoint.


**Data Version:** v4 class is filtered to include only direct suicidal confessions


## Section 1: Imports

In [1]:
import pandas as pd
import numpy as np
import re
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import RobertaTokenizerFast, RobertaForSequenceClassification
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
)
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')


## Section 2: Load `suicidewatch_filtered_df` and minimal preprocessing

In [2]:
suicidewatch_filtered_df=pd.read_csv("final_suicidewatch_filtered_dataset.csv")

In [3]:
# suicidewatch_filtered_df is expected to already exist in the kernel with columns:
#   other_posts, multiclass_label, threeclass_label
# If you need to load it from disk, uncomment the line below:
# suicidewatch_filtered_df = pd.read_csv('suicidewatch_filtered_df.csv')

data = suicidewatch_filtered_df.sample(frac=1, random_state=42).reset_index(drop=True)

print('Dataset shape:', data.shape)
print('\nColumns:', data.columns.tolist())
print('\nthreeclass_label distribution:')
print(data['threeclass_label'].value_counts(dropna=False))
print('\nmulticlass_label distribution:')
print(data['multiclass_label'].value_counts(dropna=False))

Dataset shape: (106279, 17)

Columns: ['author', 'subreddit', 'report_post', 'other_posts', 'is_self_report', 'is_control', 'self_report_sentence', 'selfdiag_score', 'selfdiag_matches', 'severe_flag', 'label', 'multiclass_label', 'threeclass_label', 'clean_text', 'matches', 'score', 'decision']

threeclass_label distribution:
threeclass_label
at_risk     80140
normal      22688
suicidal     3451
Name: count, dtype: int64

multiclass_label distribution:
multiclass_label
control          22688
adhd             17915
anxiety          14437
depression       13078
mentalhealth     11529
socialanxiety     4299
bpd               4178
ptsd              3792
suicidewatch      3451
autism            2375
schizophrenia     1897
healthanxiety     1879
bipolarreddit     1457
edanonymous       1398
addiction          752
lonely             623
alcoholism         531
Name: count, dtype: int64


In [4]:
def minimal_preprocess(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

data['other_posts'] = data['other_posts'].apply(minimal_preprocess)
data = data[data['other_posts'].str.len() > 0].reset_index(drop=True)
print(f'After cleaning empty text: {len(data)} rows')

After cleaning empty text: 105906 rows


## Section 3: Shared building blocks (Dataset, train, eval)

In [5]:
class TextClassificationDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = torch.tensor(np.asarray(labels), dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item


def train_epoch(model, train_loader, optimizer, scheduler, device, loss_fn):
    model.train()
    total_loss = 0
    for batch in tqdm(train_loader, desc='Training'):
        batch = {k: v.to(device) for k, v in batch.items()}
        labels = batch.pop('labels')
        outputs = model(**batch)
        loss = loss_fn(outputs.logits, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(train_loader)


def evaluate(model, eval_loader, device, loss_fn):
    model.eval()
    total_loss = 0
    predictions, true_labels = [], []
    with torch.no_grad():
        for batch in tqdm(eval_loader, desc='Evaluating'):
            batch = {k: v.to(device) for k, v in batch.items()}
            labels = batch.pop('labels')
            outputs = model(**batch)
            loss = loss_fn(outputs.logits, labels)
            total_loss += loss.item()
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            predictions.extend(preds)
            true_labels.extend(labels.cpu().numpy())
    return total_loss / len(eval_loader), accuracy_score(true_labels, predictions), predictions, true_labels


In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

MODEL_NAME = 'roberta-base'
tokenizer = RobertaTokenizerFast.from_pretrained(MODEL_NAME)
MAX_LENGTH = 256
BATCH_SIZE = 16
NUM_EPOCHS = 3
LEARNING_RATE = 2e-5

Using device: cuda


## Section 4: End-to-end training function

Given a dataframe + label column, this prepares splits, encodes labels, computes class weights, trains a RoBERTa model, and reports macro / weighted P/R/F1 plus a confusion matrix. We call it twice — once per label column.

In [7]:
def run_experiment(df, label_col, model_save_path, min_samples_per_class=50):
    print('=' * 70)
    print(f'EXPERIMENT: {label_col}  ->  {model_save_path}')
    print('=' * 70)

    # 1. Drop NaN labels / empty text
    work = df.dropna(subset=[label_col, 'other_posts']).copy()

    # 2. Drop ultra-rare classes
    counts = work[label_col].value_counts()
    keep_classes = counts[counts >= min_samples_per_class].index.tolist()
    dropped = counts[counts < min_samples_per_class]
    if len(dropped) > 0:
        print(f'Dropping {len(dropped)} classes with < {min_samples_per_class} samples:')
        for cls, n in dropped.items():
            print(f'  - {cls}: {n}')
    work = work[work[label_col].isin(keep_classes)].reset_index(drop=True)

    # 3. Encode string labels -> integers
    le = LabelEncoder()
    work['_y'] = le.fit_transform(work[label_col].astype(str))
    class_names = list(le.classes_)
    num_labels = len(class_names)
    print(f'\nFinal: {len(work)} rows, {num_labels} classes')
    print('Class distribution:')
    for i, name in enumerate(class_names):
        print(f'  {i:2d} {name:<25} {(work["_y"] == i).sum()}')

    # 4. 80/10/10 stratified split
    train_df, temp_df = train_test_split(work, test_size=0.2, random_state=42, stratify=work['_y'])
    val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['_y'])
    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)
    print(f'\nSplit sizes: train={len(train_df)}, val={len(val_df)}, test={len(test_df)}')

    # 5. Tokenize
    def tok(texts):
        return tokenizer(texts.tolist(), max_length=MAX_LENGTH, padding='max_length',
                         truncation=True, return_tensors='pt')

    print('\nTokenizing...')
    train_enc = tok(train_df['other_posts'])
    val_enc = tok(val_df['other_posts'])
    test_enc = tok(test_df['other_posts'])

    train_ds = TextClassificationDataset(train_enc, train_df['_y'].values)
    val_ds = TextClassificationDataset(val_enc, val_df['_y'].values)
    test_ds = TextClassificationDataset(test_enc, test_df['_y'].values)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

    # 6. Model + class-weighted loss
    id2label = {i: n for i, n in enumerate(class_names)}
    label2id = {n: i for i, n in enumerate(class_names)}
    model = RobertaForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=num_labels,
        id2label=id2label,
        label2id=label2id,
    ).to(device)

    cw = compute_class_weight('balanced', classes=np.arange(num_labels), y=train_df['_y'].values)
    class_weights = torch.tensor(cw, dtype=torch.float).to(device)
    loss_fn = nn.CrossEntropyLoss(weight=class_weights)
    print(f'\nClass weights computed (min={cw.min():.3f}, max={cw.max():.3f})')

    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
    total_steps = len(train_loader) * NUM_EPOCHS
    scheduler = torch.optim.lr_scheduler.LinearLR(optimizer, total_iters=total_steps)

    # 7. Training loop — select on macro-F1
    best_val_f1 = 0.0
    for epoch in range(NUM_EPOCHS):
        print(f'\n--- Epoch {epoch + 1}/{NUM_EPOCHS} ---')
        train_loss = train_epoch(model, train_loader, optimizer, scheduler, device, loss_fn)
        print(f'Train loss: {train_loss:.4f}')
        val_loss, val_acc, val_preds, val_true = evaluate(model, val_loader, device, loss_fn)
        val_macro_f1 = f1_score(val_true, val_preds, average='macro', zero_division=0)
        print(f'Val loss: {val_loss:.4f}  acc: {val_acc:.4f}  macro-F1: {val_macro_f1:.4f}')
        if val_macro_f1 > best_val_f1:
            best_val_f1 = val_macro_f1
            torch.save(model.state_dict(), model_save_path)
            print(f'  -> saved best model to {model_save_path}')

    # 8. Final test eval with best checkpoint
    model.load_state_dict(torch.load(model_save_path))
    test_loss, test_acc, test_preds, test_true = evaluate(model, test_loader, device, loss_fn)
    print('\n' + '=' * 70)
    print(f'TEST RESULTS — {label_col}')
    print('=' * 70)
    print(f'Loss:        {test_loss:.4f}')
    print(f'Accuracy:    {test_acc:.4f}')
    print(f'Macro    P/R/F1: '
          f'{precision_score(test_true, test_preds, average="macro", zero_division=0):.4f} / '
          f'{recall_score(test_true, test_preds, average="macro", zero_division=0):.4f} / '
          f'{f1_score(test_true, test_preds, average="macro", zero_division=0):.4f}')
    print(f'Weighted P/R/F1: '
          f'{precision_score(test_true, test_preds, average="weighted", zero_division=0):.4f} / '
          f'{recall_score(test_true, test_preds, average="weighted", zero_division=0):.4f} / '
          f'{f1_score(test_true, test_preds, average="weighted", zero_division=0):.4f}')
    print('\nClassification report:')
    print(classification_report(test_true, test_preds, target_names=class_names, zero_division=0))
    print('Confusion matrix (rows=true, cols=pred):')
    cm = confusion_matrix(test_true, test_preds, labels=list(range(num_labels)))
    print(cm)

    return {
        'model': model,
        'label_encoder': le,
        'class_names': class_names,
        'test_acc': test_acc,
        'test_macro_f1': f1_score(test_true, test_preds, average='macro', zero_division=0),
    }


## Section 5: Train Model 1 — 3-class severity (`threeclass_label`)

In [8]:
result_3class = run_experiment(
    df=data,
    label_col='threeclass_label',
    model_save_path='best_roberta_threeclass_v4.pt',
    min_samples_per_class=50,
)

EXPERIMENT: threeclass_label  ->  best_roberta_threeclass_v4.pt

Final: 105906 rows, 3 classes
Class distribution:
   0 at_risk                   79767
   1 normal                    22688
   2 suicidal                  3451

Split sizes: train=84724, val=10591, test=10591

Tokenizing...


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Class weights computed (min=0.443, max=10.229)

--- Epoch 1/3 ---


Training: 100%|██████████| 5296/5296 [40:28<00:00,  2.18it/s]


Train loss: 0.1366


...

Train loss: 0.0615


Evaluating: 100%|██████████| 662/662 [01:27<00:00,  7.57it/s]


Val loss: 0.0709  acc: 0.9845  macro-F1: 0.9309
  -> saved best model to best_roberta_threeclass_v4.pt


Evaluating: 100%|██████████| 662/662 [01:28<00:00,  7.51it/s]



TEST RESULTS — threeclass_label
Loss:        0.1012
Accuracy:    0.9807
Macro    P/R/F1: 0.8844 / 0.9626 / 0.9170
Weighted P/R/F1: 0.9848 / 0.9807 / 0.9821

Classification report:
              precision    recall  f1-score   support

     at_risk       1.00      0.98      0.99      7977
      normal       0.99      1.00      1.00      2269
    suicidal       0.66      0.91      0.77       345

    accuracy                           0.98     10591
   macro avg       0.88      0.96      0.92     10591
weighted avg       0.98      0.98      0.98     10591

Confusion matrix (rows=true, cols=pred):
[[7806   11  160]
 [   2 2267    0]
 [  30    1  314]]


## Section 6: Train Model 2 — Subreddit model (`multiclass_label`)

In [9]:
picked_classes=["adhd","anxiety","control" , "depression", "suicidewatch" ]
data = data[data['multiclass_label'].isin(picked_classes)].reset_index(drop=True)
result_multiclass = run_experiment(
    df=data,
    label_col='multiclass_label',
    model_save_path='best_roberta_multiclass_v4.pt',
    # min_samples_per_class=12000,  # drops jokes (6), covid19_support (37)
)

EXPERIMENT: multiclass_label  ->  best_roberta_multiclass_v4.pt

Final: 71369 rows, 5 classes
Class distribution:
   0 adhd                      17875
   1 anxiety                   14401
   2 control                   22688
   3 depression                12954
   4 suicidewatch              3451

Split sizes: train=57095, val=7137, test=7137

Tokenizing...


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Class weights computed (min=0.629, max=4.136)

--- Epoch 1/3 ---


Training: 100%|██████████| 3569/3569 [27:34<00:00,  2.16it/s]


Train loss: 0.5302


...

Train loss: 0.2250


Evaluating: 100%|██████████| 447/447 [00:58<00:00,  7.64it/s]


Val loss: 0.2486  acc: 0.9306  macro-F1: 0.9186
  -> saved best model to best_roberta_multiclass_v4.pt


Evaluating: 100%|██████████| 447/447 [00:58<00:00,  7.60it/s]



TEST RESULTS — multiclass_label
Loss:        0.2436
Accuracy:    0.9344
Macro    P/R/F1: 0.9272 / 0.9221 / 0.9240
Weighted P/R/F1: 0.9365 / 0.9344 / 0.9349

Classification report:
              precision    recall  f1-score   support

        adhd       0.96      0.90      0.93      1788
     anxiety       0.87      0.92      0.89      1440
     control       1.00      0.99      1.00      2269
  depression       0.86      0.90      0.88      1295
suicidewatch       0.94      0.89      0.92       345

    accuracy                           0.93      7137
   macro avg       0.93      0.92      0.92      7137
weighted avg       0.94      0.93      0.93      7137

Confusion matrix (rows=true, cols=pred):
[[1613  117    0   56    2]
 [  26 1324    0   90    0]
 [   2    2 2256    9    0]
 [  28   82    0 1168   17]
 [   3    4    0   30  308]]


## Section 7: Summary

In [ ]:
print('=' * 60)
print('FINAL SUMMARY — RoBERTa')
print('=' * 60)
print(f"3-class model    | classes={len(three_class_names)}    | "
      f"test acc={result_3class['test_acc']:.4f} | macro-F1={result_3class['test_macro_f1']:.4f}")
print(f"Subreddit model  | classes={len(class_names)}   | "
      f"test acc={result_multiclass['test_acc']:.4f} | macro-F1={result_multiclass['test_macro_f1']:.4f}")

FINAL SUMMARY — RoBERTa
3-class model    | classes=3    | test acc=0.9807 | macro-F1=0.9170
Subreddit model  | classes=5   | test acc=0.9344 | macro-F1=0.9240


In [8]:
def predict_single(text, model_path, num_labels, class_names):
    """Predict the class of a single text input using a trained RoBERTa checkpoint."""
    # Rebuild model architecture and load weights
    id2label = {i: n for i, n in enumerate(class_names)}
    label2id = {n: i for i, n in enumerate(class_names)}
    model = RobertaForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=num_labels,
        id2label=id2label,
        label2id=label2id,
    ).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    # Preprocess + tokenize
    clean = minimal_preprocess(text)
    enc = tokenizer(
        clean,
        max_length=MAX_LENGTH,
        padding='max_length',
        truncation=True,
        return_tensors='pt',
    ).to(device)

    # Forward pass
    with torch.no_grad():
        logits = model(**enc).logits
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]

    pred_idx = int(probs.argmax())
    pred_label = class_names[pred_idx]

    print(f'Input: {text[:120]}{"..." if len(text) > 120 else ""}')
    print(f'\nPredicted class: {pred_label}  (confidence: {probs[pred_idx]:.4f})')
    print('\nTop 5 probabilities:')
    top5 = np.argsort(probs)[::-1][:5]
    for i in top5:
        print(f'  {class_names[i]:<25} {probs[i]:.4f}')

    return pred_label, probs

In [9]:
three_class_names=["at_risk","normal","suicidal"]
class_names=["adhd","anxiety","control","depression","suicidewatch"]

In [10]:
# Example text
sample_text = "I've been feeling really overwhelmed lately, can't sleep, and everything feels pointless. I don't know what to do anymore."

# Predict with the 3-class severity model
print('=== 3-class severity model ===')
predict_single(
    text=sample_text,
    model_path='best_roberta_threeclass.pt',
    num_labels=len(three_class_names),
    class_names=three_class_names,
)

print('\n=== Subreddit (multiclass) model ===')
predict_single(
    text=sample_text,
    model_path='best_roberta_multiclass_v4.pt',
    num_labels=len(class_names),
    class_names=class_names,
)

=== 3-class severity model ===


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Input: I've been feeling really overwhelmed lately, can't sleep, and everything feels pointless. I don't know what to do anymor...

Predicted class: suicidal  (confidence: 0.9429)

Top 5 probabilities:
  suicidal                  0.9429
  normal                    0.0508
  at_risk                   0.0064

=== Subreddit (multiclass) model ===


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Input: I've been feeling really overwhelmed lately, can't sleep, and everything feels pointless. I don't know what to do anymor...

Predicted class: adhd  (confidence: 0.7708)

Top 5 probabilities:
  adhd                      0.7708
  anxiety                   0.0686
  suicidewatch              0.0650
  depression                0.0644
  control                   0.0312


('adhd',
 array([0.77083534, 0.06860758, 0.03124378, 0.06436085, 0.06495252],
       dtype=float32))

In [11]:
# Example text
sample_text = "I've been feeling really overwhelmed lately."

# Predict with the 3-class severity model
print('=== 3-class severity model ===')
predict_single(
    text=sample_text,
    model_path='best_roberta_threeclass.pt',
    num_labels=len(three_class_names),
    class_names=three_class_names,
)

print('\n=== Subreddit (multiclass) model ===')
predict_single(
    text=sample_text,
    model_path='best_roberta_multiclass_v4.pt',
    num_labels=len(class_names),
    class_names=class_names,
)

=== 3-class severity model ===


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Input: I've been feeling really overwhelmed lately.

Predicted class: suicidal  (confidence: 0.8684)

Top 5 probabilities:
  suicidal                  0.8684
  normal                    0.1225
  at_risk                   0.0090

=== Subreddit (multiclass) model ===


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Input: I've been feeling really overwhelmed lately.

Predicted class: adhd  (confidence: 0.8195)

Top 5 probabilities:
  adhd                      0.8195
  control                   0.0723
  anxiety                   0.0597
  suicidewatch              0.0323
  depression                0.0161


('adhd',
 array([0.819525  , 0.05973935, 0.07234181, 0.01611827, 0.03227563],
       dtype=float32))

In [12]:
# Example text
sample_text = "I feel so empty inside. I don't know what to do anymore."

# Predict with the 3-class severity model
print('=== 3-class severity model ===')
predict_single(
    text=sample_text,
    model_path='best_roberta_threeclass.pt',
    num_labels=len(three_class_names),
    class_names=three_class_names,
)

print('\n=== Subreddit (multiclass) model ===')
predict_single(
    text=sample_text,
    model_path='best_roberta_multiclass_v4.pt',
    num_labels=len(class_names),
    class_names=class_names,
)

=== 3-class severity model ===


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Input: I feel so empty inside. I don't know what to do anymore.

Predicted class: suicidal  (confidence: 0.9417)

Top 5 probabilities:
  suicidal                  0.9417
  normal                    0.0534
  at_risk                   0.0048

=== Subreddit (multiclass) model ===


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Input: I feel so empty inside. I don't know what to do anymore.

Predicted class: adhd  (confidence: 0.6990)

Top 5 probabilities:
  adhd                      0.6990
  control                   0.1662
  anxiety                   0.0772
  suicidewatch              0.0320
  depression                0.0256


('adhd',
 array([0.6989508 , 0.07719972, 0.16621046, 0.02561674, 0.03202228],
       dtype=float32))

In [13]:
# Example text
sample_text = "I don't know want to live anymore."

# Predict with the 3-class severity model
print('=== 3-class severity model ===')
predict_single(
    text=sample_text,
    model_path='best_roberta_threeclass.pt',
    num_labels=len(three_class_names),
    class_names=three_class_names,
)

print('\n=== Subreddit (multiclass) model ===')
predict_single(
    text=sample_text,
    model_path='best_roberta_multiclass_v4.pt',
    num_labels=len(class_names),
    class_names=class_names,
)

=== 3-class severity model ===


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Input: I don't know want to live anymore.

Predicted class: suicidal  (confidence: 0.9595)

Top 5 probabilities:
  suicidal                  0.9595
  normal                    0.0367
  at_risk                   0.0038

=== Subreddit (multiclass) model ===


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Input: I don't know want to live anymore.

Predicted class: control  (confidence: 0.6326)

Top 5 probabilities:
  control                   0.6326
  adhd                      0.2616
  anxiety                   0.0525
  suicidewatch              0.0328
  depression                0.0205


('control',
 array([0.26155406, 0.0524666 , 0.6326373 , 0.0205325 , 0.03280958],
       dtype=float32))

In [14]:
# Example text
sample_text = "I'm fine."

# Predict with the 3-class severity model
print('=== 3-class severity model ===')
predict_single(
    text=sample_text,
    model_path='best_roberta_threeclass.pt',
    num_labels=len(three_class_names),
    class_names=three_class_names,
)

print('\n=== Subreddit (multiclass) model ===')
predict_single(
    text=sample_text,
    model_path='best_roberta_multiclass_v4.pt',
    num_labels=len(class_names),
    class_names=class_names,
)

=== 3-class severity model ===


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Input: I'm fine.

Predicted class: normal  (confidence: 0.5880)

Top 5 probabilities:
  normal                    0.5880
  suicidal                  0.3643
  at_risk                   0.0477

=== Subreddit (multiclass) model ===


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Input: I'm fine.

Predicted class: control  (confidence: 0.9592)

Top 5 probabilities:
  control                   0.9592
  adhd                      0.0180
  anxiety                   0.0156
  depression                0.0044
  suicidewatch              0.0028


('control',
 array([0.01798056, 0.01557282, 0.9592154 , 0.00440182, 0.00282945],
       dtype=float32))

In [15]:
# Example text
sample_text = "I'm fine.life seems okay.everything is good.just a bit stressed with work. so i'm a little worried about that."

# Predict with the 3-class severity model
print('=== 3-class severity model ===')
predict_single(
    text=sample_text,
    model_path='best_roberta_threeclass.pt',
    num_labels=len(three_class_names),
    class_names=three_class_names,
)

print('\n=== Subreddit (multiclass) model ===')
predict_single(
    text=sample_text,
    model_path='best_roberta_multiclass_v4.pt',
    num_labels=len(class_names),
    class_names=class_names,
)

=== 3-class severity model ===


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Input: I'm fine.life seems okay.everything is good.just a bit stressed with work. so i'm a little worried about that.

Predicted class: suicidal  (confidence: 0.6960)

Top 5 probabilities:
  suicidal                  0.6960
  normal                    0.2928
  at_risk                   0.0112

=== Subreddit (multiclass) model ===


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Input: I'm fine.life seems okay.everything is good.just a bit stressed with work. so i'm a little worried about that.

Predicted class: adhd  (confidence: 0.6218)

Top 5 probabilities:
  adhd                      0.6218
  anxiety                   0.1296
  control                   0.1123
  depression                0.0973
  suicidewatch              0.0391


('adhd',
 array([0.6217943 , 0.12955654, 0.11226375, 0.09726633, 0.03911907],
       dtype=float32))

In [16]:
# Example text
sample_text = "I'm fine.life seems okay.everything is good.just a bit stressed with work. so i'm a little worried about that."

# Predict with the 3-class severity model
print('=== 3-class severity model ===')
predict_single(
    text=sample_text,
    model_path='best_roberta_threeclass.pt',
    num_labels=len(three_class_names),
    class_names=three_class_names,
)

print('\n=== Subreddit (multiclass) model ===')
predict_single(
    text=sample_text,
    model_path='best_roberta_multiclass_v4.pt',
    num_labels=len(class_names),
    class_names=class_names,
)

=== 3-class severity model ===


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Input: I'm fine.life seems okay.everything is good.just a bit stressed with work. so i'm a little worried about that.

Predicted class: suicidal  (confidence: 0.6960)

Top 5 probabilities:
  suicidal                  0.6960
  normal                    0.2928
  at_risk                   0.0112

=== Subreddit (multiclass) model ===


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Input: I'm fine.life seems okay.everything is good.just a bit stressed with work. so i'm a little worried about that.

Predicted class: adhd  (confidence: 0.6218)

Top 5 probabilities:
  adhd                      0.6218
  anxiety                   0.1296
  control                   0.1123
  depression                0.0973
  suicidewatch              0.0391


('adhd',
 array([0.6217943 , 0.12955654, 0.11226375, 0.09726633, 0.03911907],
       dtype=float32))

In [17]:
# Example text
sample_text = "i can't foucs on anything latly , it's like my brain is just not working.\
i can't even read a book without losing focus after a few sentences."

# Predict with the 3-class severity model
print('=== 3-class severity model ===')
predict_single(
    text=sample_text,
    model_path='best_roberta_threeclass.pt',
    num_labels=len(three_class_names),
    class_names=three_class_names,
)

print('\n=== Subreddit (multiclass) model ===')
predict_single(
    text=sample_text,
    model_path='best_roberta_multiclass_v4.pt',
    num_labels=len(class_names),
    class_names=class_names,
)

=== 3-class severity model ===


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Input: i can't foucs on anything latly , it's like my brain is just not working.i can't even read a book without losing focus a...

Predicted class: suicidal  (confidence: 0.7090)

Top 5 probabilities:
  suicidal                  0.7090
  normal                    0.2734
  at_risk                   0.0177

=== Subreddit (multiclass) model ===


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Input: i can't foucs on anything latly , it's like my brain is just not working.i can't even read a book without losing focus a...

Predicted class: adhd  (confidence: 0.5696)

Top 5 probabilities:
  adhd                      0.5696
  anxiety                   0.2321
  depression                0.1278
  suicidewatch              0.0609
  control                   0.0096


('adhd',
 array([0.5696363 , 0.23211983, 0.00956583, 0.12782224, 0.06085584],
       dtype=float32))

In [18]:
# Example text
sample_text = "my brain is not letting me relax , i'm always worried about something\
    though i know can't fix everything but i can't stop overthinking" 


# Predict with the 3-class severity model
print('=== 3-class severity model ===')
predict_single(
    text=sample_text,
    model_path='best_roberta_threeclass.pt',
    num_labels=len(three_class_names),
    class_names=three_class_names,
)

print('\n=== Subreddit (multiclass) model ===')
predict_single(
    text=sample_text,
    model_path='best_roberta_multiclass_v4.pt',
    num_labels=len(class_names),
    class_names=class_names,
)

=== 3-class severity model ===


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Input: my brain is not letting me relax , i'm always worried about something    though i know can't fix everything but i can't ...

Predicted class: suicidal  (confidence: 0.7953)

Top 5 probabilities:
  suicidal                  0.7953
  normal                    0.1662
  at_risk                   0.0385

=== Subreddit (multiclass) model ===


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Input: my brain is not letting me relax , i'm always worried about something    though i know can't fix everything but i can't ...

Predicted class: adhd  (confidence: 0.4913)

Top 5 probabilities:
  adhd                      0.4913
  depression                0.1948
  suicidewatch              0.1607
  anxiety                   0.1432
  control                   0.0101


('adhd',
 array([0.4912674 , 0.14321198, 0.01007495, 0.19479038, 0.16065529],
       dtype=float32))

In [19]:
# Example text
sample_text = "i'm feeling like i've lost passion on everything, i don't enjoy anything anymore \
                it does not make a difference for me if it is a small or big thing, i just don't care anymore." 


# Predict with the 3-class severity model
print('=== 3-class severity model ===')
predict_single(
    text=sample_text,
    model_path='best_roberta_threeclass.pt',
    num_labels=len(three_class_names),
    class_names=three_class_names,
)

print('\n=== Subreddit (multiclass) model ===')
predict_single(
    text=sample_text,
    model_path='best_roberta_multiclass_v4.pt',
    num_labels=len(class_names),
    class_names=class_names,
)

=== 3-class severity model ===


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Input: i'm feeling like i've lost passion on everything, i don't enjoy anything anymore                 it does not make a diff...

Predicted class: suicidal  (confidence: 0.8066)

Top 5 probabilities:
  suicidal                  0.8066
  normal                    0.1750
  at_risk                   0.0184

=== Subreddit (multiclass) model ===


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Input: i'm feeling like i've lost passion on everything, i don't enjoy anything anymore                 it does not make a diff...

Predicted class: adhd  (confidence: 0.5272)

Top 5 probabilities:
  adhd                      0.5272
  anxiety                   0.1518
  control                   0.1388
  depression                0.1074
  suicidewatch              0.0748


('adhd',
 array([0.52720904, 0.15177506, 0.13884032, 0.10740832, 0.07476723],
       dtype=float32))

In [20]:
# Example text
sample_text = "i don't know how i feel, work is overloading me , and i feel prusure having to cop with everything , work, family and fun\
     but i can't denay that i'm greatful to the life i have  " 


# Predict with the 3-class severity model
print('=== 3-class severity model ===')
predict_single(
    text=sample_text,
    model_path='best_roberta_threeclass.pt',
    num_labels=len(three_class_names),
    class_names=three_class_names,
)

print('\n=== Subreddit (multiclass) model ===')
predict_single(
    text=sample_text,
    model_path='best_roberta_multiclass_v4.pt',
    num_labels=len(class_names),
    class_names=class_names,
)

=== 3-class severity model ===


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Input: i don't know how i feel, work is overloading me , and i feel prusure having to cop with everything , work, family and fu...

Predicted class: suicidal  (confidence: 0.9061)

Top 5 probabilities:
  suicidal                  0.9061
  normal                    0.0704
  at_risk                   0.0235

=== Subreddit (multiclass) model ===


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Input: i don't know how i feel, work is overloading me , and i feel prusure having to cop with everything , work, family and fu...

Predicted class: adhd  (confidence: 0.5350)

Top 5 probabilities:
  adhd                      0.5350
  suicidewatch              0.1856
  depression                0.1637
  anxiety                   0.1021
  control                   0.0136


('adhd',
 array([0.53497154, 0.10205532, 0.01363173, 0.16371426, 0.18562718],
       dtype=float32))

In [21]:
# Example text
sample_text = "the most annoying thing that i can't really take care of the people i love\
           i'm not taking enough care of my family and boyfrind  " 


# Predict with the 3-class severity model
print('=== 3-class severity model ===')
predict_single(
    text=sample_text,
    model_path='best_roberta_threeclass.pt',
    num_labels=len(three_class_names),
    class_names=three_class_names,
)

print('\n=== Subreddit (multiclass) model ===')
predict_single(
    text=sample_text,
    model_path='best_roberta_multiclass_v4.pt',
    num_labels=len(class_names),
    class_names=class_names,
)

=== 3-class severity model ===


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Input: the most annoying thing that i can't really take care of the people i love           i'm not taking enough care of my fa...

Predicted class: suicidal  (confidence: 0.7711)

Top 5 probabilities:
  suicidal                  0.7711
  normal                    0.2041
  at_risk                   0.0248

=== Subreddit (multiclass) model ===


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Input: the most annoying thing that i can't really take care of the people i love           i'm not taking enough care of my fa...

Predicted class: adhd  (confidence: 0.6086)

Top 5 probabilities:
  adhd                      0.6086
  suicidewatch              0.2034
  depression                0.0965
  anxiety                   0.0714
  control                   0.0200


('adhd',
 array([0.6086291 , 0.07143357, 0.02001552, 0.09654023, 0.20338151],
       dtype=float32))